# Open-Loop Reconfiguration Test

Run a scripted sequence of R/T/N burns through `RecfgEnv` and plot the ROE evolution to sanity-check axis mapping and reward shaping.

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from leo_gym.gyms.recfg_gym import RecfgEnv

from train_recfg_hppo_cfg import sat_cfg, env_cfg

In [2]:

def run_scripted_plan(env: RecfgEnv, plan, reset_seed: int | None = None):
    """Execute a list of (axis_id, delay_steps, duration_steps) actions."""
    obs_history, reward_history, info_history, n_history = [], [], [], []
    if reset_seed is not None:
        obs, info = env.reset(seed=reset_seed)
    else:
        obs, info = env.reset()
    obs_history.append(obs)
    info_history.append(info)
    n_history.append(info.get("n", getattr(env.satellite, "discrete_time_index_simulation", 0)))

    for axis_id, delay, dur in plan:
        action = {"discrete": axis_id, "continuous": np.array([delay, dur], dtype=np.float64)}
        obs, reward, terminated, truncated, info = env.step(action)
        obs_history.append(obs)
        reward_history.append(reward)
        info_history.append(info)
        n_history.append(info.get("n", getattr(env.satellite, "discrete_time_index_simulation", 0)))
        if terminated or truncated:
            break

    return {
        "obs_history": np.array(obs_history),
        "reward_history": np.array(reward_history),
        "info_history": info_history,
        "n_history": np.array(n_history),
    }



def plot_roe(history: np.ndarray, n_history: np.ndarray, dt_seconds: float) -> None:
    t = np.array(n_history) * dt_seconds / 60.0  # minutes from sim index
    labels = ['a·δa', 'a·δλ', 'a·δe_x', 'a·δe_y', 'a·δi_x', 'a·δi_y']
    fig = make_subplots(rows=3, cols=2, shared_xaxes=True, subplot_titles=labels)
    for i, label in enumerate(labels):
        if i >= history.shape[1]:
            break
        row = i // 2 + 1
        col = i % 2 + 1
        fig.add_trace(go.Scatter(x=t, y=history[:, i], mode='lines', name=label), row=row, col=col)
    fig.update_layout(height=800, width=900, title_text='ROE evolution (interactive)', showlegend=False)
    fig.update_xaxes(title_text='Time [min]', row=3, col=1)
    fig.update_xaxes(title_text='Time [min]', row=3, col=2)
    fig.show()


In [3]:
# Set SEED to an int for reproducible runs, or None for random each time.
SEED = 42

# axis_id map from RecfgEnv: 
# 0:+R,
# 1:-R,
# 2:+T,
# 3:-T,
# 4:+N,
# 5:-N,
# 6:coast

env_seed = SEED if SEED is not None else np.random.randint(0, 2**32 - 1)
env = RecfgEnv(env_cfg, seed=env_seed)


scripted = [
    (4, 0, 41),   # +N burn immediately
    (5, 0, 41),   # -N burn immediately
    (4, 0, 41),   # +N burn immediately
    (2, 0, 41),   # +T burn immediately
    (0, 0, 41)    # +R burn immediately

]

result = run_scripted_plan(env, scripted, reset_seed=env_seed)

In [4]:
satellite = env.satellite
satellite.plot_states_interactive()